In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/complaint-sense-consumer-complaint-classification-challenge/sample_submission.csv
/kaggle/input/competitions/complaint-sense-consumer-complaint-classification-challenge/test_complaints.csv
/kaggle/input/competitions/complaint-sense-consumer-complaint-classification-challenge/dataset-metadata.json
/kaggle/input/competitions/complaint-sense-consumer-complaint-classification-challenge/train_complaints.csv
/kaggle/input/competitions/complaint-sense-consumer-complaint-classification-challenge/baseline_submission.csv


In [2]:
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.4 MB/s eta 0:00:00


In [4]:
import torch, transformers, datasets, evaluate
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.10.0+cpu
Transformers: 5.0.0
GPU available: False


In [5]:
from datasets import load_dataset

dataset = load_dataset("csv", data_files={"train": "/kaggle/input/competitions/complaint-sense-consumer-complaint-classification-challenge/train_complaints.csv"})

Generating train split: 0 examples [00:00, ? examples/s]

In [6]:
import pandas as pd
train_csv = pd.read_csv("/kaggle/input/competitions/complaint-sense-consumer-complaint-classification-challenge/train_complaints.csv")  
print(train_csv.shape)
print(train_csv.head())

(380, 4)
   ComplaintId                                               text  \
0            1  Flagging an issue: Username change broke acces...   
1            2  Writing to complain: Blender motor smells like...   
2            3  Hello, Courier left perishable groceries in th...   
3            4  Hi support — Tablet touchscreen registers ghos...   
4            5  Hi support — Authorized service charged labor ...   

            Category                      FamilyId  
0     account_access     template:account_access:5  
1     product_defect     template:product_defect:6  
2  delivery_shipping  template:delivery_shipping:1  
3     product_defect     template:product_defect:7  
4    warranty_repair    template:warranty_repair:3  


In [7]:
# Label mapping
label_names = sorted(set(train_csv["Category"]))
label2id = {label: i for i, label in enumerate(label_names)}
id2label = {i: label for label, i in label2id.items()}
num_labels = len(label_names)

In [8]:
# Adding labels column to dataset
def encode_labels(example):
    example["labels"] = label2id[example["Category"]]
    return example

dataset = dataset.map(encode_labels)

Map:   0%|          | 0/380 [00:00<?, ? examples/s]

In [9]:
from transformers import AutoTokenizer

model_id = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_id)

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=256)

tokenized_dataset = dataset.map(tokenize_batch, batched=True)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/380 [00:00<?, ? examples/s]

In [10]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_id, num_labels=10, id2label=id2label, label2id=label2id
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
import evaluate

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits,labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="macro")["f1"]
    return {"accuracy": accuracy, "f1": f1}

In [12]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="distilbert-complaintsense",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=5,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",
    
)

In [13]:
#Splitting the train_complaints dataset
split_dataset = tokenized_dataset["train"].train_test_split(test_size=0.15, seed=42)

In [14]:
from transformers import Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset= split_dataset["train"],
    eval_dataset=split_dataset["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [15]:
trainer.train()
print(trainer.evaluate())

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,2.238340,2.184780,0.368421,0.306232
2,1.986967,1.898015,0.789474,0.751282
3,1.739855,1.638770,0.877193,0.846207
4,1.554302,1.473938,0.894737,0.864774
5,1.424964,1.411117,0.929825,0.897622


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 1.411116600036621, 'eval_accuracy': 0.9298245614035088, 'eval_f1': 0.8976219205630971, 'eval_runtime': 0.9736, 'eval_samples_per_second': 58.546, 'eval_steps_per_second': 4.108, 'epoch': 5.0}


In [16]:
print(len(split_dataset["test"]))
from collections import Counter
print(Counter(split_dataset["test"]["labels"]))

57
Counter({1: 8, 6: 7, 7: 7, 0: 7, 3: 6, 9: 6, 5: 5, 4: 5, 2: 5, 8: 1})


In [17]:
from datasets import Dataset
import torch

#Load test data
test_csv = pd.read_csv("/kaggle/input/competitions/complaint-sense-consumer-complaint-classification-challenge/test_complaints.csv")
test_dataset = Dataset.from_pandas(test_csv)

# Tokenize
def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=256)

tokenized_test = test_dataset.map(tokenize_batch, batched=True)

# Run the model in inference mode
model.eval()
predictions = []

with torch.no_grad():
    for i in range(len(tokenized_test)):
        inputs = {
            "input_ids": torch.tensor([tokenized_test[i]["input_ids"]]),
            "attention_mask": torch.tensor([tokenized_test[i]["attention_mask"]]),
        }
        outputs = model(**inputs)
        predicted_id = torch.argmax(outputs.logits, dim=-1).item()
        predictions.append(predicted_id)

# Convert integer predictions back to category strings
predicted_labels = [id2label[pred] for pred in predictions]

# Build the submission dataframe
submission = pd.DataFrame({
    "ComplaintId": test_csv["ComplaintId"],
    "Category": predicted_labels
})



submission.to_csv("/kaggle/working/submission.csv", index=False)

Map:   0%|          | 0/160 [00:00<?, ? examples/s]